In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jangedoo/utkface-new")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'utkface-new' dataset.
Path to dataset files: /kaggle/input/utkface-new


In [2]:
folder_path = '/root/.cache/kagglehub/datasets/jangedoo/utkface-new/versions/1/UTKFace'

In [3]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Dense,
    GlobalAveragePooling2D,
    RandomFlip,
    RandomRotation,
    RandomZoom
)

from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

In [4]:
# 100_0_0_20170110183726330.jpg -> age_gender_race_date.jpg

In [5]:
# Read the image filenames
age = []
gender = []
img_path = []

for file in os.listdir(folder_path):

    try:
        age.append(int(file.split('_')[0]))
        gender.append(int(file.split('_')[1]))
        img_path.append(file)

    except:
        pass

In [6]:
df = pd.DataFrame({
    'age': age,
    'gender': gender,
    'img': img_path
})

In [7]:
df.head()

,age,gender,img
0,40,0,40_0_0_20170117203219447.jpg.chip.jpg
1,26,1,26_1_3_20170119193141890.jpg.chip.jpg
2,36,1,36_1_0_20170116165722892.jpg.chip.jpg
3,1,1,1_1_0_20170109194410994.jpg.chip.jpg
4,67,1,67_1_3_20170109132529672.jpg.chip.jpg


In [8]:
# Shuffle the dataset
df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [9]:
train_df = df.iloc[:20000]
test_df = df.iloc[20000:]

In [10]:
print("Training:", len(train_df))
print("Testing:", len(test_df))

Training: 20000
Testing: 3708


In [11]:
# Data Pipeline Creation and Image Preprocessing.
# Path Construction
train_paths = [os.path.join(folder_path, x) for x in train_df['img']]
test_paths = [os.path.join(folder_path, x) for x in test_df['img']]

# Label Extraction
train_age = train_df['age'].values
train_gender = train_df['gender'].values

test_age = test_df['age'].values
test_gender = test_df['gender'].values

# TensorFlow Dataset Creation -> Converts paths and labels into a tf.data.Dataset pipeline
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_paths, (train_age, train_gender))
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (test_paths, (test_age, test_gender))
)

# Image Loading and Preprocessing
def load_image(path, labels):
    age, gender = labels

    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (224, 224))

    return image, (age, gender)

# map() applies the load_image() function to every element/sample in the dataset
train_dataset = train_dataset.map(load_image)
test_dataset = test_dataset.map(load_image)

# Batching
train_dataset = train_dataset.batch(32)
test_dataset = test_dataset.batch(32)

In [12]:
# Create data augmentation
data_augmentation = tf.keras.Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    RandomZoom(0.1)
])

In [13]:
# Applying augmentation and ResNet50 preprocessing
inputs = Input(
    shape=(224, 224, 3)
)

# Augmentation is applied before ResNet50 preprocessing
x = data_augmentation(inputs)

# ResNet50 preprocessing is applied after augmentation
x = preprocess_input(x)

In [14]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

In [15]:
base_model.trainable = False

In [16]:
# Pass image through ResNet50
x = base_model(
    x,
    training=False
)

In [17]:
# Convert feature maps to a vector(using this instead of Flatten())
x = GlobalAveragePooling2D()(x)

In [18]:
age_output = Dense(
    1,
    activation='linear',
    name='age'
)(x)

gender_output = Dense(
    1,
    activation='sigmoid',
    name='gender'
)(x)

In [19]:
# Create the Functional API model
model = Model(
    inputs=inputs,
    outputs=[
        age_output,
        gender_output
    ]
)

In [20]:
model.compile(
    optimizer='adam',

    loss={
        'age': 'mae',
        'gender': 'binary_crossentropy'
    },

    metrics={
        'age': 'mae',
        'gender': 'accuracy'
    }
)

In [21]:
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=10
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 125s 182ms/step - age_loss: 12.6081 - age_mae: 12.6081 - gender_accuracy: 0.8030 - gender_loss: 0.4260 - loss: 13.0341 - val_age_loss: 10.2768 - val_age_mae: 10.2781 - val_gender_accuracy: 0.8563 - val_gender_loss: 0.3193 - val_loss: 10.5976
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 116s 186ms/step - age_loss: 9.9142 - age_mae: 9.9142 - gender_accuracy: 0.8400 - gender_loss: 0.3622 - loss: 10.2764 - val_age_loss: 9.4971 - val_age_mae: 9.4981 - val_gender_accuracy: 0.8673 - val_gender_loss: 0.3017 - val_loss: 9.7999
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 141s 184ms/step - age_loss: 9.3875 - age_mae: 9.3875 - gender_accuracy: 0.8525 - gender_loss: 0.3406 - loss: 9.7281 - val_age_loss: 9.1117 - val_age_mae: 9.1126 - val_gender_accuracy: 0.8714 - val_gender_loss: 0.2904 - val_loss: 9.4031
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 114s 183ms/step - age_loss: 9.1289 - age_mae: 9.1289 - gender_accuracy: 0.8543 - gender_loss: 0.3299 - loss: 9.4587 - val_age